# NES-RE: regularized evolution

Evolve a population with ensemble-aware parent selection and removal of the oldest member.
Start with [Ensemble basics](../01_ensemble_basics.ipynb) and follow [setup instructions](../README.md).
This notebook is self-contained and uses synthetic regression data on CPU.

In [ ]:
import random

import matplotlib.pyplot as plt
import torch
from torch import nn
from torch.utils.data import DataLoader, TensorDataset

from bensemble.search import EvolutionarySearcher, SearchSpace, regression_mse_criterion

SEED = 42
random.seed(SEED)
torch.manual_seed(SEED)
torch.set_num_threads(1)  # Small CPU workloads avoid thread-pool overhead.
DEVICE = torch.device("cpu")
EPOCHS = 250
POOL_SIZE = 8
ENSEMBLE_SIZE = 3

## 1. Data and evaluation protocol

Generate a noisy regression problem and split it into disjoint train, validation,
and test sets. Training inputs lie in `[-3, 3]`; the plotting grid extends to
`[-5, 5]`. Test RMSE measures interpolation on fresh noisy targets. The wider grid
illustrates extrapolation, without assuming that uncertainty must grow outside
the training range.

Candidate training uses only the train split. Selection uses only validation; test data stay untouched until final evaluation.

In [ ]:
def true_function(x):
    return torch.sin(1.5 * x) + 0.2 * x


generator = torch.Generator().manual_seed(SEED)
features = 6 * torch.rand(300, 1, generator=generator) - 3
targets = true_function(features) + 0.15 * torch.randn(300, 1, generator=generator)
indices = torch.randperm(len(features), generator=generator)
split_indices = {
    "train": indices[:180],
    "validation": indices[180:240],
    "test": indices[240:],
}
x_train, y_train = features[split_indices["train"]], targets[split_indices["train"]]
x_val, y_val = (
    features[split_indices["validation"]],
    targets[split_indices["validation"]],
)
x_test, y_test = features[split_indices["test"]], targets[split_indices["test"]]
x_grid = torch.linspace(-5, 5, 240).reshape(-1, 1)

plt.figure(figsize=(8, 4))
plt.scatter(x_train[:, 0], y_train[:, 0], s=12, label="Train")
plt.plot(x_grid[:, 0], true_function(x_grid)[:, 0], "k--", label="True function")
plt.axvspan(-3, 3, alpha=0.08, color="green", label="Training input range")
plt.xlabel("x")
plt.ylabel("y")
plt.legend()
plt.show()

## 2. Define the architecture space and candidate training

The space varies hidden width and depth. A mutation changes one choice without
modifying the parent configuration. Seeds cover both Python `random` and PyTorch.
Repeated architecture configurations are allowed: their trained weights differ.

Each candidate uses the same training budget. Examples share this small setup so
the search APIs are easy to compare; they are not a multi-seed method benchmark.

In [ ]:
class ToySearchSpace(SearchSpace):
    WIDTHS = (16, 32, 64)
    DEPTHS = (1, 2)

    def sample(self):
        return {
            "width": random.choice(self.WIDTHS),
            "depth": random.choice(self.DEPTHS),
        }

    def mutate(self, config):
        child = dict(config)
        key = random.choice(["width", "depth"])
        choices = self.WIDTHS if key == "width" else self.DEPTHS
        child[key] = random.choice([value for value in choices if value != config[key]])
        return child

    def build(self, config):
        layers = []
        in_features = 1
        for _ in range(config["depth"]):
            layers.extend([nn.Linear(in_features, config["width"]), nn.Tanh()])
            in_features = config["width"]
        layers.append(nn.Linear(in_features, 1))
        return nn.Sequential(*layers)


def train_fn(model):
    model.to(DEVICE).train()
    optimizer = torch.optim.Adam(model.parameters(), lr=0.01)
    for _ in range(EPOCHS):
        optimizer.zero_grad()
        loss = nn.functional.mse_loss(model(x_train), y_train)
        loss.backward()
        optimizer.step()
    model.eval()


val_loader = DataLoader(TensorDataset(x_val, y_val), batch_size=64, shuffle=False)

## 3. Search using held-out validation data

Pass `regression_mse_criterion` explicitly: the default search criterion is for
classification. The final ensemble averages regression outputs directly.

With population size 4 and pool size 8, the search trains 4 initial candidates
and 4 mutated children. Parent selection uses 2 ensemble-compatible candidates;
the final 3-member selection considers the entire training history.

In [ ]:
searcher = EvolutionarySearcher(
    space=ToySearchSpace(),
    train_fn=train_fn,
    pool_size=POOL_SIZE,
    ensemble_size=ENSEMBLE_SIZE,
    population_size=4,
    num_parent_candidates=2,
    criterion=regression_mse_criterion,
    device=DEVICE,
)
ensemble = searcher.search(val_loader).eval()
print(f"Selected {ensemble.num_members} members from {POOL_SIZE} trained candidates")

## 4. Evaluate once on test data

Report test RMSE and plot the individual member curves alongside the mixture mean.
The band measures model disagreement; it excludes observation noise and is not
a calibrated 95% interval. Similar models may produce a narrow band even where
their predictions are wrong.

In [ ]:
with torch.no_grad():
    member_predictions = ensemble.predict_members(x_grid)
    mean = member_predictions.mean(dim=0)
    std = member_predictions.std(dim=0)
    test_mean = ensemble(x_test)

assert member_predictions.shape == (ENSEMBLE_SIZE, len(x_grid), 1)
assert torch.isfinite(member_predictions).all()
test_rmse = (test_mean - y_test).square().mean().sqrt().item()
print(f"Held-out test RMSE: {test_rmse:.3f}")

In [ ]:
plt.figure(figsize=(9, 4))
plt.scatter(x_train[:, 0], y_train[:, 0], s=12, alpha=0.5, label="Train")
plt.plot(x_grid[:, 0], true_function(x_grid)[:, 0], "k--", label="True function")
plt.plot(x_grid[:, 0], mean[:, 0], label="Predictive mean")
plt.fill_between(
    x_grid[:, 0],
    (mean - 2 * std)[:, 0],
    (mean + 2 * std)[:, 0],
    alpha=0.2,
    label="Mean ± 2 model standard deviations",
)
plt.axvline(-3, color="grey", linestyle=":")
plt.axvline(3, color="grey", linestyle=":")
plt.xlabel("x")
plt.ylabel("y")
plt.legend()
plt.show()

In [ ]:
plt.figure(figsize=(9, 4))
for i, predictions in enumerate(member_predictions):
    plt.plot(x_grid[:, 0], predictions[:, 0], alpha=0.5, label=f"Member {i + 1}")
plt.plot(x_grid[:, 0], mean[:, 0], "k--", linewidth=2, label="Ensemble mean")
plt.xlabel("x")
plt.ylabel("y")
plt.legend()
plt.show()

[Compare NES-RS](nes_rs.ipynb), [NES-RE](nes_re.ipynb), or [Bayesian sampling](nes_bayesian_sampling.ipynb).